### Fix KenLM model and save model with LM support


In [1]:
with open(
    "/home/davud/Desktop/storage/0x/hf/kenlm/build/bin/5gram.arpa",
    "r",
    encoding="ISO-8859-1",
) as read_file, open(
    "/home/davud/Desktop/storage/0x/hf/kenlm/build/bin/5gram.correct.arpa",
    "w",
    encoding="utf-8",
) as write_file:
    has_added_eos = False
    for line in read_file:
        if not has_added_eos and "ngram 1=" in line:
            count = line.strip().split("=")[-1]
            write_file.write(line.replace(f"{count}", f"{int(count) + 1}"))
        elif not has_added_eos and "<s>" in line:
            write_file.write(line)
            write_file.write(line.replace("<s>", "</s>"))
            has_added_eos = True
        else:
            write_file.write(line)


In [ ]:
from transformers import AutoProcessor
from pyctcdecode import build_ctcdecoder
from transformers import Wav2Vec2ProcessorWithLM

processor = AutoProcessor.from_pretrained("./model")
vocab_dict = processor.tokenizer.get_vocab()
sorted_vocab_dict = {
    k.lower(): v for k, v in sorted(vocab_dict.items(), key=lambda item: item[1])
}
print(sorted_vocab_dict)

decoder = build_ctcdecoder(
    labels=list(sorted_vocab_dict.keys()),
    kenlm_model_path="kenlm/build/bin/5gram.correct.arpa",
)

processor_with_lm = Wav2Vec2ProcessorWithLM(
    feature_extractor=processor.feature_extractor,
    tokenizer=processor.tokenizer,
    decoder=decoder,
)

processor_with_lm.save_pretrained("./withlm")

### Test


In [1]:
# Load model directly
from transformers import AutoProcessor, AutoModelForCTC
import librosa
import torch

processor = AutoProcessor.from_pretrained("./withlm")
model = AutoModelForCTC.from_pretrained("./withlm")

In [2]:
def infer(file: str, alpha=0.10584604317636571, beta=3.556553344365402):
    audio, sr = librosa.load(
        file,
        sr=16000,
    )
    input_values = processor(audio, sampling_rate=sr, return_tensors="pt").to(
        model.device
    )
    with torch.no_grad():
        logits = model(**input_values).logits

    logits = logits[0].cpu().numpy()  # Remove batch dimension and convert to NumPy
    return processor.decode(logits, alpha=alpha, beta=beta)


infer("/home/davud/Desktop/storage/webspeech/azeri-test/audio/f2_ID4_58_book1.wav")

Wav2Vec2DecoderWithLMOutput(text='عاغیلسیز دوستان عاغیللی دوشمان یاخشی دیر', logit_score=-2.8775882257143, lm_score=-5.616824973366777, word_offsets=None)

### Smoketest


In [5]:
import json
import os
from tqdm import tqdm

with open(
    "/home/davud/Desktop/storage/webspeech/azeri-test/data.json", "r", encoding="utf-8"
) as f:
    data = json.load(f)

results = []
for i in tqdm(data[:10]):
    result = infer(
        os.path.join(
            "/home/davud/Desktop/storage/webspeech/azeri-test/audio", i["audio"]
        )
    )
    results.append(
        {
            "audio": i["audio"],
            "text": result.text,
            "logit_score": result.logit_score,
            "lm_score": result.lm_score,
        }
    )

100%|██████████| 10/10 [00:06<00:00,  1.62it/s]


In [6]:
results

[{'audio': 'm1_ID889_12452_book4.wav',
  'text': 'هر کس او ایلانی اونون بوغاجیندان آچدی قیزی اونا وئر جیم',
  'logit_score': -2.7618187257898237,
  'lm_score': -7.612054229682621},
 {'audio': 'f3_ID400_5597_book3.wav',
  'text': 'پادا ا تانا ب اوام گتیره بیلمز',
  'logit_score': -128.87484424383837,
  'lm_score': -132.0362807427389},
 {'audio': 'o2_ID788_11059_book4.wav',
  'text': 'لالا قاچیب بیر سو قیلیفینده گیزلندی',
  'logit_score': -3.373409209044489,
  'lm_score': -6.112645956696966},
 {'audio': 'f6_ID233_3268_book2.wav',
  'text': 'سوسن خان اوغلانی یالقیز گؤروب یانینا گلدی اونون بئله آوارا اولماسی نین سیرینی اوندان سوروشدو',
  'logit_score': -2.3642565949625016,
  'lm_score': -8.903291103847554},
 {'audio': 'f2_ID884_12363_book4.wav',
  'text': 'پادشاه دئمیشدی کی قیرخ گونه تاپدین تاپدین',
  'logit_score': -0.6296215129537884,
  'lm_score': -3.79105801185433},
 {'audio': 'o2_ID129_1820_book1.wav',
  'text': 'صهل ابن مالک لغمانین یانینا گلیب قیزین قارداشی ایله دانیشدی',
  'logit_s